# Data Ingestion from external sources

# Why the Bronze Layer Should Not Communicate with the Internet
1. Separation of concerns
External data fetching and internal data processing have distinct responsibilities and should be handled by separate jobs.
2. Deterministic processing
Bronze jobs become fully reproducible since they rely only on immutable internal data, not changing API responses.
3. Replayability and backfills
Any Bronze table can be rebuilt from raw files without re-calling external APIs.
4. Improved reliability
API outages, rate limits, or network issues do not affect Bronze, Silver, or Gold layers.
5. Security and access control
Internet access and API credentials are restricted to ingestion jobs, reducing attack surface and risk.
6. Easier testing and CI/CD
Bronze jobs can be tested with static input data and do not require API keys or network access.
7. Clear data lineage
The flow from external source to internal tables is explicit and auditable.
8. Scalability and performance
Large reprocessing jobs scale on Spark without stressing or re-hitting external APIs.
9. Vendor independence
Downstream logic remains unchanged if data providers are added, removed, or replaced.
10. Alignment with industry best practices
This design follows medallion architecture and real-world enterprise data platform standards.

# Install


In [0]:
from datetime import datetime
import os
import pandas as pd
import yfinance as yf

# Define

In [0]:
BASE_PATH = "/Volumes/workspace/finance_tracking_stocks/stock_data/yahoo/ingestion"
INGESTION_DATE = datetime.utcnow().strftime("%Y-%m-%d")

In [0]:
### Yahoo fetch function (pure Python)
def fetch_yahoo(tickers, start="2000-01-01"):
    df = yf.download(
        tickers=tickers,
        start=start,
        group_by="ticker",
        auto_adjust=False,
        threads=True,
        progress=False
    )

    records = []

    for ticker in tickers:
        if ticker not in df.columns.levels[0]:
            continue
        tmp = df[ticker].reset_index()
        tmp["ticker"] = ticker
        records.append(tmp)

    if not records:
        return pd.DataFrame()

    out = pd.concat(records)
    out.columns = [c.lower().replace(" ", "_") for c in out.columns]
    return out

# Scaling using Spark

In [0]:
### Step 1: Load ticker universe
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]  # later 1000+

In [0]:
### Step 2: Chunk tickers
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

ticker_chunks = list(chunk_list(tickers, 50))


In [0]:
### Step 3: Parallel ingestion via Spark
# Spark here is an orchestrator = not actually doing transformations :
# ticker_chunks → Spark tasks → Yahoo API → Parquet files in Volume
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext  # sc = SparkContext, which is the lower-level object for parallel operations. (parallelism engine)
# uusually call sc.parallelize() and sc.stop() : On Databricks, don’t usually stop it manually — it’s managed by the cluster.

def ingest_chunk(chunk):
    pdf = fetch_yahoo(chunk)
    if pdf.empty:
        return

    output_dir = f"{BASE_PATH}/ingestion_date={INGESTION_DATE}"
    os.makedirs(output_dir, exist_ok=True)
    # checks if the folder exists at the specified path (output_dir). If not, it creates it. 
    # The exist_ok=True argument ensures no error is raised if the folder already exists. 
    # This approach works for any path in Databricks volumes, including nested folders.

    file_name = f"yahoo_{datetime.utcnow().strftime('%H%M%S_%f')}.parquet"
    pdf.to_parquet(f"{output_dir}/{file_name}", index=False)  # index=False -> Don’t include Pandas row index in Parquet file.
    # By default, Pandas writes a column for the row numbers (0,1,2…). -> Don’t want that in raw data.

sc.parallelize(ticker_chunks).foreach(ingest_chunk)
### sc.parallelize(ticker_chunks)
# Creates an RDD (Resilient Distributed Dataset) from the list of chunks.
# Each element of the list becomes one Spark task -> Each task will run on a different executor (worker node) if the cluster has multiple nodes.
### .foreach(ingest_chunk)
# Calls the function on each RDD element in parallel.
# Each worker executes ingest_chunk(chunk) independently.

# Serverless Databricks vs. classic cluster
* Classic Spark cluster:
* You get worker nodes + a driver
* You can access spark.sparkContext (sc) and do sc.parallelize()
* Each element of an RDD runs on a worker node
* Serverless Databricks cluster:
* No persistent Spark driver JVM you can control
* The “driver” is managed and hidden
* Certain low-level Spark operations that require direct access to JVM objects, like sc.parallelize(), are not allowed
* DataFrame operations still work because they run on the managed execution engine


# 1️⃣ You can parallelize on serverless — but not using sc.parallelize()
* Serverless Spark still has workers and a distributed engine.
* Spark automatically parallelizes DataFrame operations (transformations, aggregations, joins, writes, etc.) across workers.
* What you cannot do is manually create RDDs and directly control task distribution with sc.parallelize() because the underlying driver JVM is hidden.

So the parallelism exists, but it’s controlled by Spark’s execution engine, not by your Python calls to sc.parallelize().

⸻

# 2️⃣ Purpose of Spark on serverless

Even if you can’t manually distribute RDD tasks:
1. Automatic parallelism for DataFrames / SQL
  * df.groupBy().agg()
  * df.join(df2)
  * df.write.format("delta").mode("append")
Spark splits these operations into tasks across multiple workers automatically.
2. Scalability
  * Handles large datasets that cannot fit in a single machine memory.
3. Fault tolerance & retries
  * If a task fails on a worker, Spark reruns it automatically.
4. Optimized execution
  * Catalyst optimizer chooses the best physical plan.
  * Serverless auto-scales workers for large workloads.

⸻

# 3️⃣ Why sc.parallelize() is different
* sc.parallelize() is manual task distribution at the Python/RDD level.
* Serverless doesn’t give you direct access to the driver JVM, so you can’t use that low-level API
* Spark wants you to rely on DataFrames for distributed operations instead of manual RDD orchestration.

# 1️⃣ Pandas vs Spark DataFrames
* pdf in your code is a Pandas DataFrame (pd.DataFrame).
  * It lives in Python, in the driver’s memory.
  * Operations on it (like pdf.to_parquet()) happen locally, not distributed across Spark workers.
* A Spark DataFrame (spark.read(...) or spark.createDataFrame(...)) lives in the Spark engine.
  * Transformations (select, filter, groupBy, join) are automatically split across workers.
  * This is where parallelism happens natively in Databricks Spark.

So yes, pdf is a DataFrame, but not a Spark DataFrame → serverless Spark doesn’t parallelize it.

⸻

# 2️⃣ Serverless analogy

Perfect analogy:

Serverless Spark is like renting a fully automated car:
* You can drive it (run DataFrame operations, queries, writes)
* You cannot open the engine and manually rearrange pistons (sc.parallelize())

In contrast, a classic Spark cluster is like owning a car where you can also tweak the engine (manual RDD orchestration).

⸻

# 3️⃣ Implications for your Yahoo ingestion job
* fetch_yahoo returns Pandas DF → local memory → not parallelized by Spark
* On serverless, sc.parallelize() doesn’t work → can’t manually distribute tasks
* Solution:
  * Use Python threads (or async IO) for parallel API calls
	* Once data is written to volume, you can read it as Spark DataFrame in Bronze layer → then you get full distributed parallelism

# Saving files

Python threading : 
* Runs on driver only
* Can write to any path driver sees
* Single-threaded file writes
* /Volumes/... works locally


Spark :
* Runs on driver + multiple worker nodes
* Must write to shared storage accessible by all workers
* Parallel writes by partitions
* /Volumes/... does not exist on workers (serverless)
